In [ ]:
import numpy as np
import xarray as xr
import cftime
import dask
import matplotlib.pyplot as plt
import os
import cftime
import xesmf as xe
from os.path import exists
import cesmesptools

In [ ]:
d = xr.open_dataset('/glade/work/jtcohen/SMYLE_anom_drift_trend.TEMP.11.1989-2018.nc')
d['TEMP'].isel(Y=29).mean(('nlat', 'nlon', 'M'))

## Open Dask

In [ ]:
import dask_jobqueue
import distributed

# this first part is checking you're in the right environment
if "client" in locals():
    client.close()
    del client
if "cluster" in locals():
    cluster.close()

# this is where we set up the cluster, your own compute system if you will 
cluster = dask_jobqueue.PBSCluster(
    cores=1,  # The number of cores you want
    memory="15GB",  # Amount of memory
    processes=1,  # How many processes
    queue="casper",  # The type of queue to utilize (/glade/u/apps/dav/opt/usr/bin/execcasper)
    # log_directory="/glade/scratch/dcherian/dask/",  # Use your local directory
    resource_spec="select=1:ncpus=1:mem=15GB",  # Specify resources
    account="uwis0040",  # Input your project ID here / THIS WILL BE DIFFERENT FOR YOU 
    walltime="00:20:00",  # Amount of wall time
    interface="ext",  # Interface to use
)

# this is where we say that we want several of these compute systems,
# because we will have to deal with lots of data and can't just rely on one
cluster.adapt(maximum_jobs=24, minimum_jobs=2) # If you want to force everything to be quicker, 
# increase the number of minimum jobs, but sometimes then it will take a while until you get them assigned 
# (they have to queue), so it's a trade-off
client = distributed.Client(cluster)

In [ ]:
client

## Load Data

In [ ]:
field = 'TEMP' # 'TEMP' 'PSL'
component = 'ocn' # 'ocn' 'atm'
model = '.pop.h.' # '.pop.h.' '.cam.h0.'
datadir = '/glade/campaign/cesm/development/espwg/SMYLE/archive/'
casename = 'b.e21.BSMYLE.f09_g17.????-MM.EEE'
filetemplate = datadir+casename+'/'+component+'/proc/tseries/month_1/'+casename+model+field+'.*.nc'
#check out this diretory to see what's all available in SMYLE

ens = np.arange(20)
firstyear = 1989
lastyear  = 2018

month_names = {
    2: 'February',
    5: 'May',
    8: 'August',
    11: 'November'
}
# startmonth = 2  # 2 for Feb, 5 for May, 8 for Aug, 11 for Nov

#define chunks here
chunks={'z_t':1,'L':24,}

## User tasks go here 
def preprocess(ds):
    # return ds.isel({'nlat':slice(160,240)})
    # return ds.isel({'nlat':slice(250,320), 'nlon':slice(200, 250)})
    return ds.isel({'z_t': 0})

## Remove Drift and Trend

In [ ]:
def remove_trend(da, dim, deg=1):
    # detrend along a single dimension
    # return polyfit coefficients and detrended da
    p = da.polyfit(dim=dim, deg=deg, skipna=True)
    coord = da.coords[dim]
    fit = xr.polyval(coord, p.polyfit_coefficients)
    return p.polyfit_coefficients, da - fit


def remove_drift_trend(startmonth):
    """
    Calculates SMYLE anomalies with respect to the drift and linear trend across ensembles.
    Loads SMYLE data, removes the climatological drift, then detrends the anomalies by lead time.
    """
    smyle_tmp = cesmesptools.get_monthly_data(filetemplate, ens, field, firstyear, lastyear, startmonth,\
                                           preprocess, chunks=chunks, client=client)
    print(f'size in GB: {(smyle_tmp.nbytes/1e9):0.2f}') #GB
    da = smyle_tmp[field]

    time_bound = smyle_tmp.time_bound.load()
    d1 = cftime.DatetimeNoLeap(firstyear,1,1,0,0,0)
    d2 = cftime.DatetimeNoLeap(lastyear,12,31,23,59,59)

    masked_period = da.where((time_bound.mean('d2')>d1) & (time_bound.mean('d2')<d2))
    climodrift = masked_period.mean(dim=['M', 'Y'])
    no_drift = da - climodrift

    p = no_drift.mean(dim='M').polyfit(dim='Y', deg=1, skipna=True)
    coord = no_drift.coords['Y']
    ensemble_mean_trend = xr.polyval(coord, p.polyfit_coefficients)

    da_anom = no_drift - ensemble_mean_trend

    return da_anom, climodrift, ensemble_mean_trend, p.polyfit_coefficients

In [ ]:
%%time
starts = [5]
da_anoms = {}
climodrifts = {}
trends = {}
coeffs = {}

for startmonth in starts:
    da_anoms[startmonth], climodrifts[startmonth], trends[startmonth], coeffs[startmonth] = remove_drift_trend(startmonth)

# Save

In [ ]:
%%time

outdir = '/glade/work/jtcohen/'

for startmonth in starts:
    ds = xr.Dataset({'TEMP': da_anoms[startmonth],
                     'drift': climodrifts[startmonth],
                     'trend': trends[startmonth],
                     'coeffs': coeffs[startmonth]})
    
    fout = f'SMYLE_anom_drift_trend.{field}.{startmonth:02d}.{firstyear}-{lastyear}.nc'
    
    if exists(outdir+fout): 
        print('File already exists. Are you sure you want to recalculate?')
    else:
        print(f'Saving start month: {startmonth}')
        ds.load().to_netcdf(outdir+fout, engine='netcdf4')
        print(f'file saved at {outdir+fout}')
        ds.close()